# Maestro v2: el Maestro ahora mira el mercado

**Quién lo corre:** Paolo (PS). **No** cambia nada de las 90 corridas: es una mejora que se presenta aparte.

**Qué arregla:** en el sistema del Sprint 6, el Maestro aprendió con la volatilidad y la actividad del mercado fijas y el desbalance en 0, y sin la corrección del cierre. Aquí aprende de nuevo:
- midiendo la volatilidad, la profundidad del libro y el desbalance en el mercado simulado (lo que vio el comprador en la ventana anterior);
- con la corrección del cierre también al entrenar;
- contando sus decisiones sobre las que tiene cada orden.

Los compradores (Ejecutores) son los mismos del sistema final.

**Cómo usarlo:** «Entorno de ejecución → Ejecutar todas». Dura **≈ 3 horas** en total (entrenar ≈ 40 min y evaluar ≈ 2 h). Si Colab se corta, vuelve a «Ejecutar todas»: retoma donde quedó.

**Al final:** la celda 14 dice **LISTO**, la 15 compara el Maestro nuevo con el anterior y con TWAP, y la 16 hace el zip `maestro_v2_media.zip`. Ese zip se lo pasas a Claude.

**Necesita en tu Drive:** `slippage_colab/resultados/sistema_final/ejecutor_elegido.pt` y la carpeta `slippage_colab/resultados/corridas_90/media/` (ya las tienes).

## Preparación del entorno (celdas 1 a 9)

In [ ]:
# CELDA 1 (reinicia el entorno solo; el aviso "sesión fallada" es normal, no la repitas)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# CELDA 2
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
!conda run -n abides python --version

In [ ]:
# CELDA 3 (unos minutos; un aviso de Pillow es normal)
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# CELDA 4 (deben salir tres "OK")
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO"; tail -20 /content/install_{pkg}.log)

In [ ]:
# CELDA 5 (debe imprimir 0.18.0 1.22.0 1.2.4 0.29.1 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# CELDA 6 (torch; debe imprimir torch 1.13.1+cpu | numpy 1.22.0)
!conda run -n abides python -m pip install -q torch==1.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!conda run -n abides python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

In [ ]:
# CELDA 7 (código del proyecto: rama feature/ps-maestro-v2-ve-mercado, o main si ya se mergeó; debe mostrar un commit y un número mayor que 0)
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && (git checkout -q -f origin/feature/ps-maestro-v2-ve-mercado || git checkout -q -f origin/main)
!cd /content/slippage && git log --oneline -1 && grep -c "sm_mercado" scripts/colab/evaluar_sistema_ps.py


In [ ]:
# CELDA 8 (conecta Drive; acepta el permiso)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/slippage_colab/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print(RESULTS_DIR)

In [ ]:
# CELDA 9 (seaborn, que necesita el módulo de benchmarks)
!conda run -n abides python -m pip install -q seaborn==0.12.2 --no-deps

## Entrenar

In [ ]:
# CELDA 10: entrenar el Maestro v2 en Poisson (≈ 40 min). Si ya terminó, no hace nada
import os, json
M2 = f'{RESULTS_DIR}/maestro_v2'
r = f'{M2}/resumen_maestro.json'
if os.path.exists(r) and 'entrenado' in json.load(open(r, encoding='utf-8')):
    print('Ya estaba entrenado:', json.load(open(r, encoding='utf-8'))['entrenado']['agregado'])
else:
    get_ipython().system(f"cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/entrenar_maestro.py --jornadas 2000 --impacto --inicio-aleatorio --lambda-penalty 0.02 --q-total 88300 --volumen-dia 882768 --completar-al-cierre --sm-mercado --ckpt-ejecutor {RESULTS_DIR}/sistema_final/ejecutor_elegido.pt --out-dir {M2} 2>&1 | grep -v UserWarning | tail -6")


## Evaluar en la liquidez media

In [ ]:
# CELDA 11: evaluar el Maestro v2 en la liquidez media, órdenes MO-01, MO-02, MO-03, MO-04, MO-05 (15 corridas, ≈ 1 h)
get_ipython().system(f"cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/evaluar_sistema_ps.py --meta-ordenes data/meta_ordenes/meta_ordenes_v2_candidata.json --entorno abides --seeds 3 --semilla-inicial 4 --escenario media --escenarios-json data/scenarios/escenarios_liquidez_v3.json --impacto --completar-al-cierre --sm-mercado --modo muestreo --ckpt-maestro {M2}/checkpoint_last.pt --ckpt-ejecutor apertura={M2}/ejecutor_apertura.pt --ckpt-ejecutor media_jornada={M2}/ejecutor_media_jornada.pt --ckpt-ejecutor cierre={M2}/ejecutor_cierre.pt --nombre PPO_sistema_v2 --max-minutes 330 --ordenes MO-01 MO-02 MO-03 MO-04 MO-05 --out-dir {M2}/eval_media/A 2>&1 | grep -v UserWarning")


In [ ]:
# CELDA 12: evaluar el Maestro v2 en la liquidez media, órdenes MO-06, MO-07, MO-08, MO-09, MO-10 (15 corridas, ≈ 1 h)
get_ipython().system(f"cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/evaluar_sistema_ps.py --meta-ordenes data/meta_ordenes/meta_ordenes_v2_candidata.json --entorno abides --seeds 3 --semilla-inicial 4 --escenario media --escenarios-json data/scenarios/escenarios_liquidez_v3.json --impacto --completar-al-cierre --sm-mercado --modo muestreo --ckpt-maestro {M2}/checkpoint_last.pt --ckpt-ejecutor apertura={M2}/ejecutor_apertura.pt --ckpt-ejecutor media_jornada={M2}/ejecutor_media_jornada.pt --ckpt-ejecutor cierre={M2}/ejecutor_cierre.pt --nombre PPO_sistema_v2 --max-minutes 330 --ordenes MO-06 MO-07 MO-08 MO-09 MO-10 --out-dir {M2}/eval_media/B 2>&1 | grep -v UserWarning")


## Revisar

In [ ]:
# CELDA 13: decisiones del Maestro nuevo (antes elegía 50 % en el 98 % de los casos)
import glob
from collections import Counter
alfas = Counter()
for f in glob.glob(f'{M2}/eval_media/*/*.json'):
    for c in json.load(open(f, encoding='utf-8')).get('corridas', {}).values():
        for d in c.get('decisiones', []):
            alfas[5 * (d['alpha_idx'] + 1)] += 1
tot = sum(alfas.values())
for a in sorted(alfas):
    print(f'{a:3d} % de lo que falta: {alfas[a]:4d} veces ({100 * alfas[a] / tot:.0f} %)')


In [ ]:
# CELDA 14: revisión (deben salir 30 corridas del Maestro v2, con cumplimiento >= 0,95)
n, malas = 0, []
for f in glob.glob(f'{M2}/eval_media/*/*.json'):
    for c in json.load(open(f, encoding='utf-8')).get('corridas', {}).values():
        if 'error' in c:
            malas.append((c['orden'], c['seed'], c['error'][:80])); continue
        n += 1
        if c.get('execution_rate', 1.0) < 0.95:
            malas.append((c['orden'], c['seed'], f"cumplimiento {c['execution_rate']:.3f}"))
print('Corridas:', n, '(se esperan 30) | problemas:', malas if malas else 'ninguno')
print('LISTO' if n == 30 and not malas else 'FALTA ALGO: vuelve a ejecutar las celdas 11 y 12 (retoman donde quedaron)')


## Comparar con el Maestro anterior

In [ ]:
# CELDA 15: Maestro v2 contra el Maestro anterior (PPO_sistema), TWAP y VWAP, en los mismos días
media90 = f'{RESULTS_DIR}/corridas_90/media'
get_ipython().system(f"cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/comparacion_estadistica_3_2_3.py --entradas {media90}/A {media90}/B {M2}/eval_media --sistema PPO_sistema_v2 --referencias PPO_sistema TWAP VWAP --out-dir {M2}/analisis --titulo 'Maestro v2 en la liquidez media' 2>&1 | grep -v Warning | head -25")


## Zip

In [ ]:
# CELDA 16: zip para mandar
import zipfile
archivos = [os.path.join(c, a) for c, _, fs in os.walk(M2) for a in fs if a.endswith(('.json', '.csv', '.md', '.png'))]
zpath = f'{RESULTS_DIR}/maestro_v2_media.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a, os.path.relpath(a, RESULTS_DIR))
print(len(archivos), 'archivos -> zip listo:', zpath, '(descárgalo desde slippage_colab/resultados en Drive)')
